# Track My Subs — אבן דרך 1
**מגישות:** עופרי כהן, מיכל זטלאוי

המחברת משכפלת את המאגר, מתקינה תלויות, מריצה בדיקות ומדגימה את הסוכן ואת השוואת ההקשרים. הקוד נמצא בקובצי Python במאגר. ההדגמה משתמשת בנתונים סינתטיים ובמודל מדומה כברירת מחדל; אם מוגדר מפתח Gemini, תתבצע גם הדגמה אמיתית שעשויה להיות כרוכה בעלות.

In [ ]:
# שכפול המאגר והתקנת תלויות
from pathlib import Path
import os
import subprocess
import sys
import tempfile

REPO_URL = "https://github.com/ofricohen14/track-my-subs-agent.git"
# להגשה ניתן לקבע את מזהה ה-commit כאן בהרצה, ללא שינוי בקובץ.
COMMIT = ""
repo_dir = Path(tempfile.mkdtemp(prefix="track-my-subs-")) / "repo"
subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
if COMMIT:
    subprocess.run(["git", "checkout", "--detach", COMMIT], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, str(repo_dir))

In [ ]:
# הרצת כל הבדיקות בפקודה אחת, ללא API
subprocess.run([sys.executable, "-m", "pytest", "-v"], check=True)

In [ ]:
# שלב 3: הדגמת לולאת הסוכן (Mock כברירת מחדל, Gemini אמיתי כשמוגדר GEMINI_API_KEY ב-Colab Secrets)
import os
from src.agent import GeminiDecisionModel, MockDecisionModel, SubscriptionAgent
from src.state import PersistentMemoryStore, SessionStore, SubscriptionRecord
from src.tools import ToolExecutor, TransientError

decision_model = MockDecisionModel()
print("Mode: MockDecisionModel")

# הרצה 1: תרחיש רגיל — חלופות ל-Netflix
sessions, mem = SessionStore(), PersistentMemoryStore()
mem.upsert_subscription('user_001', SubscriptionRecord(
    subscription_id='sub_netflix', service_name='Netflix', amount=69.90, category='streaming'
))
agent = SubscriptionAgent(sessions, mem, ToolExecutor(mem), model=decision_model)
res_normal = agent.run('t_normal', 'user_001', 'תראה לי חלופות למנוי NETFLIX שלי', scenario_name='normal_run')
print('\n--- הרצה תקינה ---')
print('סטטוס:', res_normal['status'], '| צעדים:', res_normal['trace']['steps_used'], '| מצב מודל:', res_normal['trace']['model_mode'])
print('תשובה:', res_normal['response'])

# הרצה 2: תרחיש כשל זמני (2 כשלים זמניים -> ניסיונות חוזרים -> התאוששות)
calls = {'n': 0}
def flaky_catalog(tool_name, attempt):
    if tool_name == 'search_alternatives':
        calls['n'] += 1
        if calls['n'] <= 2:
            raise TransientError(f'Mock 503 temporary failure #{calls["n"]}')

mem2 = PersistentMemoryStore()
mem2.upsert_subscription('user_002', SubscriptionRecord(
    subscription_id='sub_gym', service_name='Holmes Place Gym', amount=250.00, category='fitness'
))
agent2 = SubscriptionAgent(SessionStore(), mem2, ToolExecutor(mem2, fault_injector=flaky_catalog), model=MockDecisionModel())
res_transient = agent2.run('t_transient', 'user_002', 'הצג לי חלופות למנוי חדר הכושר שלי', retries=2, scenario_name='transient_failure')
print('\n--- הרצה עם כשל זמני והתאוששות ---')
print('סטטוס:', res_transient['status'], '| ניסיונות בכלי החיפוש:', res_transient['trace']['tool_logs'][0]['attempts'])
print('תשובה:', res_transient['response'])


In [ ]:
# שלב 4: הרצת ניסוי ההשוואה בין הקשר מלא להקשר מצומצם
from src.state import run_context_comparison
results = run_context_comparison()
for r in results:
    print(f"{r['strategy_label']} | הודעות: {r['context_messages']} | תווים: {r['chars']} | טוקנים (הערכה): {r['estimated_tokens']} ({r['pct_of_full']}%) | נכונות: {r['answer_correctness']}")
    print(f"   תשובה בפועל: {r['actual_answer']}")
    print(f"   מה אבד: {r['what_is_lost']}\n")


In [ ]:
# הדגמה נוספת עם מודל אמיתי, רק כאשר מוגדר מפתח
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
if api_key:
    mem_real = PersistentMemoryStore()
    mem_real.upsert_subscription("user_003", SubscriptionRecord(
        subscription_id="sub_netflix", service_name="Netflix", amount=69.90, category="streaming"
    ))
    real_agent = SubscriptionAgent(SessionStore(), mem_real, ToolExecutor(mem_real),
                                   model=GeminiDecisionModel(api_key=api_key))
    real_result = real_agent.run("t_real", "user_003", "תראה לי חלופות למנוי Netflix שלי")
    print("סטטוס:", real_result["status"])
    print("תשובה:", real_result["response"])
else:
    print("לא הוגדר מפתח Gemini; ההדגמה המדומה מספיקה להרצה ללא API.")